In [3]:
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

In [4]:
def write_to_submit_file(predicted_labels, out_file, target="target", index_label="session_id"):
    predicted_df = pd.DataFrame(predicted_labels, 
                                index=np.arange(1, predicted_labels.shape[0]+1),
                                columns=[target])
    predicted_df.to_csv(out_file, index_label=index_label)

In [5]:
train_df = pd.read_csv("./train_sessions.csv/train_sessions.csv", index_col="session_id")
test_df = pd.read_csv("./test_sessions.csv/test_sessions.csv", index_col="session_id")

In [6]:
train_df.columns

Index(['site1', 'time1', 'site2', 'time2', 'site3', 'time3', 'site4', 'time4',
       'site5', 'time5', 'site6', 'time6', 'site7', 'time7', 'site8', 'time8',
       'site9', 'time9', 'site10', 'time10', 'target'],
      dtype='object')

In [7]:
# convert time1, ...., time10 columns to datetime type

# times = ["time%s" % i for i in range (1,11)]
# times = ["time{}".format(i) for i in range(1, 11)]
times = [f"time{i}" for i in range(1, 11)]
train_df[times] = train_df[times].apply(pd.to_datetime)
test_df[times] = test_df[times].apply(pd.to_datetime)

# sort the data by time
train_df = train_df.sort_values(by="time1")

train_df.head()

,site1,time1,site2,time2,site3,time3,site4,time4,site5,time5,...,time6,site7,time7,site8,time8,site9,time9,site10,time10,target
session_id,,,,,,,,,,,,,,,,,,,,,
21669,56,2013-01-12 08:05:57,55.0,2013-01-12 08:05:57,NaN,NaT,NaN,NaT,NaN,NaT,...,NaT,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0
54843,56,2013-01-12 08:37:23,55.0,2013-01-12 08:37:23,56.0,2013-01-12 09:07:07,55.0,2013-01-12 09:07:09,NaN,NaT,...,NaT,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0
77292,946,2013-01-12 08:50:13,946.0,2013-01-12 08:50:14,951.0,2013-01-12 08:50:15,946.0,2013-01-12 08:50:15,946.0,2013-01-12 08:50:16,...,2013-01-12 08:50:16,948.0,2013-01-12 08:50:16,784.0,2013-01-12 08:50:16,949.0,2013-01-12 08:50:17,946.0,2013-01-12 08:50:17,0
114021,945,2013-01-12 08:50:17,948.0,2013-01-12 08:50:17,949.0,2013-01-12 08:50:18,948.0,2013-01-12 08:50:18,945.0,2013-01-12 08:50:18,...,2013-01-12 08:50:18,947.0,2013-01-12 08:50:19,945.0,2013-01-12 08:50:19,946.0,2013-01-12 08:50:19,946.0,2013-01-12 08:50:20,0
146670,947,2013-01-12 08:50:20,950.0,2013-01-12 08:50:20,948.0,2013-01-12 08:50:20,947.0,2013-01-12 08:50:21,950.0,2013-01-12 08:50:21,...,2013-01-12 08:50:21,946.0,2013-01-12 08:50:21,951.0,2013-01-12 08:50:22,946.0,2013-01-12 08:50:22,947.0,2013-01-12 08:50:22,0


In [8]:
# Transform the data into format that can be fed to CountVectorizer
sites = [f"site{i}" for i in range (1,11)] 

train_df[sites].fillna(0).astype('int').to_csv("train_session_text.txt", sep=" ", index=None, header=None)
test_df[sites].fillna(0).astype('int').to_csv("test_session_text.txt", sep=" ", index=None, header=None)

In [9]:
with open("train_session_text.txt") as f:
    for _ in range(5):
        print(f.readline().rstrip())

56 55 0 0 0 0 0 0 0 0
56 55 56 55 0 0 0 0 0 0
946 946 951 946 946 945 948 784 949 946
945 948 949 948 945 946 947 945 946 946
947 950 948 947 950 952 946 951 946 947


In [10]:
cv = CountVectorizer()
# it will be called sparse metrix
# call todense to call the full metrix
X_sparse = cv.fit_transform(["this movie is awful",
                  "enjoyed this movie"])
X_sparse.todense()

matrix([[1, 0, 1, 1, 1],
        [0, 1, 0, 1, 1]])

In [11]:
cv.vocabulary_
# this is the last one on the matrix, awful is the first one 

{'this': 4, 'movie': 3, 'is': 2, 'awful': 0, 'enjoyed': 1}

In [12]:
X_sparse.indices
X_sparse.data
X_sparse.nonzero() # -> shows the row and column indices of non zero elements

(array([0, 0, 0, 0, 1, 1, 1], dtype=int32),
 array([4, 3, 2, 0, 4, 3, 1], dtype=int32))

In [13]:
with open("train_session_text.txt") as inp_train_file:
    X_train = cv.fit_transform(inp_train_file)
with open("test_session_text.txt") as inp_test_file:
    X_test = cv.transform(inp_test_file)

print(X_train.shape, X_test.shape)

(253561, 41592) (82797, 41592)


In [14]:
y_train = train_df["target"].astype("int")

### Train logistic regression

In [15]:
logit = LogisticRegression(C=1, random_state=17)

In [16]:
cv_scores = cross_val_score(logit, X_train, y_train, cv=5, scoring="roc_auc")
cv_scores

array([0.91603556, 0.8431174 , 0.87907924, 0.89093035, 0.91099186])

In [17]:
cv_scores.mean()

np.float64(0.8880308813650295)

In [18]:
logit.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",17
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multicl

In [19]:
test_pred_logit1 = logit.predict_proba(X_test)[:,1]

No. In a logistic regression model, `logit.predict_proba(X_test)[:,1]` returns the **predicted probabilities** of belonging to the positive class (class `1`), not the model weights.

For example:

```python
probs = logit.predict_proba(X_test)
```

might produce:

```python
array([
    [0.90, 0.10],
    [0.25, 0.75],
    [0.02, 0.98]
])
```

where:

* Column 0 = (P(y=0 | x))
* Column 1 = (P(y=1 | x))

So:

```python
logit.predict_proba(X_test)[:,1]
```

returns:

```python
array([0.10, 0.75, 0.98])
```

These are the predicted probabilities for the positive class.

### If you want the model weights (coefficients)

Use:

```python
logit.coef_
```

Example:

```python
print(logit.coef_)
print(logit.intercept_)
```

The logistic regression model is:

$$
P(y=1|x) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p)}}
$$

where:

* `β₁, β₂, ...` are the coefficients (weights) returned by `logit.coef_`
* `β₀` is the intercept returned by `logit.intercept_`

### Related outputs

| Method                             | Returns                         |
| ---------------------------------- | ------------------------------- |
| `logit.predict(X_test)`            | Predicted class labels (0 or 1) |
| `logit.predict_proba(X_test)`      | Probabilities for each class    |
| `logit.predict_proba(X_test)[:,1]` | Probability of class 1          |
| `logit.coef_`                      | Feature weights (coefficients)  |
| `logit.intercept_`                 | Intercept term                  |

If you're asking about "weights of the data points" (e.g., observation/sample weights), logistic regression does not return those through `predict_proba`; those would only exist if you explicitly trained with `sample_weight=`.


In [20]:
test_pred_logit1

array([3.49637074e-03, 4.94332374e-11, 5.66205295e-12, ...,
       1.00328833e-02, 4.05119249e-04, 1.37432056e-06], shape=(82797,))

In [21]:
write_to_submit_file(test_pred_logit1, "logit_sub1.txt")

### Time features
    - hour when the feature started
    - morning
    - day 
    - evening
    - night

In [24]:
def add_time_features(time1_series, X_sparse):
    hour = time1_series.apply(lambda ts: ts.hour)
    morning = ((hour >= 7)& (hour <=11)).astype("int")
    day = ((hour >= 12)& (hour <=18)).astype("int")
    evening = ((hour >= 19)& (hour <=23)).astype("int")
    night = ((hour >= 0)& (hour <=6)).astype("int")

    # hstack must be from scipy.sparse otherwise it will take much memory and model will be inefficient
    X = hstack([X_sparse, morning.values.reshape(-1,1),
                day.values.reshape(-1,1), evening.values.reshape(-1,1), night.values.reshape(-1,1)])
    
    return X

In [25]:
X_train_with_time = add_time_features(train_df["time1"].fillna(0), X_train )
X_test_with_time = add_time_features( test_df["time1"].fillna(0), X_test)

In [26]:
X_train_with_time.shape

(253561, 41596)

In [27]:
cv_scores = cross_val_score(logit, X_train_with_time, y_train, cv=5, scoring="roc_auc")
cv_scores

array([0.92253073, 0.91189024, 0.92893796, 0.94352713, 0.94651515])

In [ ]:
cv_scores.mean()
# cv score is much higher than the site one

np.float64(0.9306802415000124)

In [29]:
logit.fit(X_train_with_time, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",17
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multicl

In [30]:
test_pred_logit2 = logit.predict_proba(X_test_with_time)[:,1]

In [31]:
write_to_submit_file(test_pred_logit2, "logit_sub2.txt")